# PyTorch Training Loop

Companion notebook for Sections 6.5 and 8.2 of the MLP lecture notes.

Objectives:

- define an MLP as an `nn.Module`;
- use `DataLoader` for mini-batches;
- implement the training loop: forward, loss, backward, update;
- evaluate with `model.eval()` and `torch.no_grad()`;
- use logits correctly with `CrossEntropyLoss`.

This notebook is written to be executable even when PyTorch is not installed. PyTorch-specific cells are skipped unless `torch` is available.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)


## 1. Dependency check


In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

print('PyTorch available:', TORCH_AVAILABLE)
if not TORCH_AVAILABLE:
    print('Install PyTorch to run the training-loop cells. The remaining cells will explain the workflow and skip execution safely.')


## 2. Dataset and preprocessing

We use `load_breast_cancer`, a local scikit-learn dataset, so no network access is required. The model predicts two classes from standardized tabular features.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Train:', X_train_scaled.shape)
print('Validation:', X_val_scaled.shape)
print('Test:', X_test_scaled.shape)


## 3. TensorDataset and DataLoader

Mini-batch training estimates gradients from a batch rather than from the full dataset.


In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(42)

    X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=128, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=128, shuffle=False)

    print('Mini-batches per epoch:', len(train_loader))
else:
    print('Skipped DataLoader creation because PyTorch is not installed.')


## 4. Define the MLP

`CrossEntropyLoss` expects raw logits. Therefore the model does not include a final softmax layer.


In [ ]:
if TORCH_AVAILABLE:
    class MLP(nn.Module):
        def __init__(self, input_dim, hidden_dims, output_dim):
            super().__init__()
            layers = []
            prev_dim = input_dim
            for hidden_dim in hidden_dims:
                layers.append(nn.Linear(prev_dim, hidden_dim))
                layers.append(nn.ReLU())
                prev_dim = hidden_dim
            layers.append(nn.Linear(prev_dim, output_dim))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x)

    model = MLP(input_dim=X_train_scaled.shape[1], hidden_dims=[32, 16], output_dim=2)
    print(model)
else:
    print('Skipped model definition because PyTorch is not installed.')


## 5. Evaluation helper

Evaluation disables dropout/batchnorm training behavior and avoids unnecessary gradient tracking.


In [ ]:
if TORCH_AVAILABLE:
    def evaluate(model, loader, criterion):
        model.eval()
        losses = []
        preds = []
        targets = []
        with torch.no_grad():
            for xb, yb in loader:
                logits = model(xb)
                loss = criterion(logits, yb)
                losses.append(loss.item())
                preds.append(torch.argmax(logits, dim=1).cpu().numpy())
                targets.append(yb.cpu().numpy())
        preds = np.concatenate(preds)
        targets = np.concatenate(targets)
        return {'loss': float(np.mean(losses)), 'accuracy': accuracy_score(targets, preds)}
else:
    print('Skipped evaluation helper because PyTorch is not installed.')


## 6. Training loop

Each mini-batch performs: clear old gradients, forward pass, loss, backward pass, optimizer update.


In [ ]:
if TORCH_AVAILABLE:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = []
    for epoch in range(1, 101):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        val_metrics = evaluate(model, val_loader, criterion)
        history.append({
            'epoch': epoch,
            'train_loss': float(np.mean(train_losses)),
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
        })

        if epoch % 20 == 0:
            print(f"epoch={epoch:3d} train_loss={history[-1]['train_loss']:.4f} val_loss={history[-1]['val_loss']:.4f} val_acc={history[-1]['val_accuracy']:.3f}")

    history_df = pd.DataFrame(history)
    history_df.tail()
else:
    history_df = pd.DataFrame()
    print('Skipped training because PyTorch is not installed.')


In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss curves')
    axes[0].legend()

    axes[1].plot(history_df['epoch'], history_df['val_accuracy'])
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation accuracy')
    axes[1].set_title('Validation accuracy')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped plots because PyTorch is not installed.')


## 7. Final test evaluation

The test set is used once, after training choices have been made.


In [ ]:
if TORCH_AVAILABLE:
    test_metrics = evaluate(model, test_loader, criterion)
    print('Test metrics:', test_metrics)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_pred = torch.argmax(test_logits, dim=1).cpu().numpy()
        test_prob = torch.softmax(test_logits, dim=1).cpu().numpy()

    print(classification_report(y_test, test_pred, target_names=load_breast_cancer().target_names))
    print('First five class probabilities:')
    print(test_prob[:5])
else:
    print('Skipped final evaluation because PyTorch is not installed.')


## 8. Why no softmax in the model?

`CrossEntropyLoss` applies `log_softmax` internally. Add softmax only at inference time if you need probabilities for reporting or downstream decisions.


## Exercises

1. Change hidden dimensions from `[32, 16]` to `[64, 32]`. Does validation accuracy improve?
2. Replace Adam with SGD. What learning rate is needed for stable training?
3. Add weight decay to Adam. Does the validation loss change?


## Takeaways

- A PyTorch training loop has three phases: forward, backward, and update.
- `loss.backward()` computes gradients; the optimizer updates parameters.
- Use `model.eval()` and `torch.no_grad()` for validation/test evaluation.
- For multi-class classification, pair raw logits with `CrossEntropyLoss`.
